# Baseline modeling with OLS and scikit-learn

The contract is now turned into two complementary problems: predicting the number of delay days (regression) and predicting whether an order will exceed the estimated date (classification). The baseline uses the same features, the same temporal split, and no actual-delivery information.

OLS is treated as an inferential and interpretable reference. Ridge (Hoerl and Kennard, 1970) is compared as a predictive regression baseline, and logistic regression (Cox, 1958) is used for `sla_breach` classification.

## Leakage-safe preparation

The modeling table attaches historical targets to the candidate table only for evaluation. Feature transformations are fitted on training and applied without refitting to validation. The Jul–Aug 2018 period was observed in earlier versions and is reserved for notebook 09.

The comparison is fair because OLS, Ridge, and logistic regression receive the same preprocessed matrix and validation period.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

LANG = "en"
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "data").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from ecom_delivery_sla import (
    baseline_feature_catalog,
    build_order_features,
    build_modeling_frame,
    build_supervised_orders,
    classification_metrics,
    fit_logistic_classification,
    fit_ols_regression,
    fit_ridge_regression,
    coefficient_table,
    fit_ols_inferential,
    ols_diagnostics,
    vif_table,
    prepare_model_matrices,
    predict_ols,
    read_olist_tables,
    regression_metrics,
    temporal_split,
)

tables = read_olist_tables(
    project_root / "data" / "raw" / "olist" / "csv",
    table_names=("orders", "items", "products", "payments", "customers", "sellers", "geolocation"),
)
supervised = build_supervised_orders(tables["orders"])
candidate_features = build_order_features(
    orders=tables["orders"],
    items=tables["items"],
    products=tables["products"],
    payments=tables["payments"],
    customers=tables["customers"],
    sellers=tables["sellers"],
    geolocation=tables["geolocation"],
)
modeling_frame = build_modeling_frame(candidate_features, supervised)
splits = temporal_split(modeling_frame, "order_purchase_timestamp", "2018-04-01", "2018-07-01")
matrices, preprocessor = prepare_model_matrices(splits)

ols_model = fit_ols_regression(matrices["train"], splits["train"]["delivery_delay_days"])
ridge_model = fit_ridge_regression(matrices["train"], splits["train"]["delivery_delay_days"])
logistic_model = fit_logistic_classification(matrices["train"], splits["train"]["sla_breach"])

display(baseline_feature_catalog())
print(f"Train rows: {len(splits['train']):,}; validation rows: {len(splits['validation']):,}")

,feature,feature_type
0,promised_lead_days,numeric
1,customer_is_capital,numeric
2,item_count,numeric
3,distinct_product_count,numeric
4,distinct_category_count,numeric
5,seller_count,numeric
6,origin_state_count,numeric
7,total_price,numeric
8,total_freight_value,numeric
9,mean_item_price,numeric


Train rows: 64,320; validation rows: 19,643


### Insight

Training ends in March 2018 and validation covers April–June. The Jul–Aug 2018 period was observed in earlier versions and is reserved for notebook 09. promised_lead_days is a candidate feature, but its interpretation depends on confirming that the promise was available at checkout.


## Predictive performance on development validation

Regression error and, for classification, AP (Davis and Goadrich, 2006; Saito and Rehmsmeier, 2015), Brier (Brier, 1950), ECE (Naeini et al., 2015) and top-k diagnostics are reported. Only development validation is used; the observed July–August 2018 period remains reserved for notebook 09.


In [2]:
regression_rows = []
classification_rows = []
for split_name in ("validation",):
    y_reg = splits[split_name]["delivery_delay_days"]
    y_cls = splits[split_name]["sla_breach"]
    ols_predictions = predict_ols(ols_model, matrices[split_name])
    ridge_predictions = ridge_model.predict(matrices[split_name])
    probabilities = logistic_model.predict_proba(matrices[split_name])[:, 1]
    for model_name, predictions in (("OLS", ols_predictions), ("Ridge", ridge_predictions)):
        regression_rows.append({"model": model_name, "split": split_name, **regression_metrics(y_reg, predictions)})
    classification_rows.append({"model": "LogisticRegression", "split": split_name, **classification_metrics(y_cls, probabilities)})

display(pd.DataFrame(regression_rows))
display(pd.DataFrame(classification_rows))

,model,split,mae,rmse,r2
0,OLS,validation,7.345099,8.835116,0.154505
1,Ridge,validation,7.342452,8.829455,0.155588


,model,split,roc_auc,average_precision,balanced_accuracy,brier,ece_10bins,precision_at_5pct,recall_at_5pct,precision_at_10pct,recall_at_10pct
0,LogisticRegression,validation,0.769379,0.116399,0.537731,0.058151,0.118445,0.15056,0.180488,0.142494,0.341463


### Insight

These numbers are development baselines, not a final ranking. The reading must consider validation performance, OLS statistical specification, and the distinction between associative interpretation and predictive capacity. The previously observed period will not be used to select features or models at this stage.


## OLS coefficients as association, not causality

OLS makes it possible to inspect conditional coefficient signs and magnitudes in the feature matrix. Because features are encoded and some are approximate, this reading must not be turned into causal claims or isolated operational effects.

The baseline opens the modeling phase, but it does not choose a production model. The next step is to diagnose residuals, classification calibration, threshold selection, and temporal stability before adding historical seller or graph variables.

The training population is used only at purchase time. The diagnostics include HC3 standard errors (White, 1980; MacKinnon and White, 1985) and seller-clustered standard errors (Cameron and Miller, 2015), VIF (Marquardt, 1970), residuals, Breusch–Pagan (Breusch and Pagan, 1979), Cook's distance (Cook, 1977), and sensitivity without absolute delays above 30 days. These are associations, not causal effects.

In [3]:
inferential_results = fit_ols_inferential(splits["train"])
ols_coefficients = coefficient_table(inferential_results["hc3"]).sort_values(
    "coefficient", key=lambda values: values.abs(), ascending=False
).head(15)
display(ols_coefficients)
display(vif_table(inferential_results["hc3"]))
diagnostics = ols_diagnostics(inferential_results["hc3"])
display(pd.DataFrame([diagnostics["breusch_pagan"]]))
display(pd.DataFrame([{"max_cooks_distance": diagnostics["max_cooks_distance"]}]))

sensitivity_frame = splits["train"].loc[
    splits["train"]["delivery_delay_days"].abs() <= 30
]
sensitivity_results = fit_ols_inferential(sensitivity_frame)
display(coefficient_table(sensitivity_results["hc3"]).head())


,term,coefficient,std_error,p_value,ci_lower,ci_upper
22,"C(customer_state, Treatment(""SP""))[T.RR]",16.304805,6.214797,8.701973e-03,4.124027,28.485584
2,"C(customer_state, Treatment(""SP""))[T.AL]",12.479030,0.695618,5.799574e-72,11.115643,13.842417
3,"C(customer_state, Treatment(""SP""))[T.AM]",11.663293,1.558476,7.221485e-14,8.608736,14.717850
4,"C(customer_state, Treatment(""SP""))[T.AP]",10.878816,3.811600,4.315432e-03,3.408218,18.349414
25,"C(customer_state, Treatment(""SP""))[T.SE]",10.559797,1.212596,3.081373e-18,8.183152,12.936441
14,"C(customer_state, Treatment(""SP""))[T.PA]",10.104859,0.542743,2.291095e-77,9.041102,11.168616
10,"C(customer_state, Treatment(""SP""))[T.MA]",9.848451,0.552701,5.055104e-71,8.765176,10.931726
6,"C(customer_state, Treatment(""SP""))[T.CE]",9.718954,0.473446,1.207213e-93,8.791018,10.646890
15,"C(customer_state, Treatment(""SP""))[T.PB]",8.206253,0.609039,2.221785e-41,7.012559,9.399948
5,"C(customer_state, Treatment(""SP""))[T.BA]",7.752731,0.276119,1.845735e-173,7.211547,8.293914


,term,vif
0,"C(customer_state, Treatment(""SP""))[T.AC]",1.010347
1,"C(customer_state, Treatment(""SP""))[T.AL]",1.022228
2,"C(customer_state, Treatment(""SP""))[T.AM]",1.021191
3,"C(customer_state, Treatment(""SP""))[T.AP]",1.011830
4,"C(customer_state, Treatment(""SP""))[T.BA]",1.115776
5,"C(customer_state, Treatment(""SP""))[T.CE]",1.060367
6,"C(customer_state, Treatment(""SP""))[T.DF]",1.041753
7,"C(customer_state, Treatment(""SP""))[T.ES]",1.046303
8,"C(customer_state, Treatment(""SP""))[T.GO]",1.056841
9,"C(customer_state, Treatment(""SP""))[T.MA]",1.034486


,lm_stat,lm_p_value,f_stat,f_p_value
0,517.098439,2.016481e-91,18.609082,7.945831e-92


,max_cooks_distance
0,0.308121


,term,coefficient,std_error,p_value,ci_lower,ci_upper
0,Intercept,1.364512,0.123141,1.553060e-28,1.123160,1.605864
1,"C(customer_state, Treatment(""SP""))[T.AC]",5.633371,1.286986,1.202243e-05,3.110924,8.155817
2,"C(customer_state, Treatment(""SP""))[T.AL]",10.937234,0.646049,2.730582e-64,9.671001,12.203466
3,"C(customer_state, Treatment(""SP""))[T.AM]",10.081818,0.929226,2.000945e-27,8.260569,11.903068
4,"C(customer_state, Treatment(""SP""))[T.AP]",5.974877,1.221965,1.010595e-06,3.579869,8.369885


### Insight

Coefficients help formulate conditional hypotheses, but they do not replace statistical diagnostics or establish causality. The model is still an initial reference and remains subject to the temporal contract of the promise.

## References

Breusch, T. S. and Pagan, A. R. (1979). A simple test for heteroscedasticity and random coefficient variation. *Econometrica*, 47(5), 1287–1294. https://doi.org/10.2307/1911963

Brier, G. W. (1950). Verification of forecasts expressed in terms of probability. *Monthly Weather Review*, 78(1), 1–3.

Cameron, A. C. and Miller, D. L. (2015). A practitioner's guide to cluster-robust inference. *Journal of Human Resources*, 50(2), 317–372. https://doi.org/10.3368/jhr.50.2.317

Cook, R. D. (1977). Detection of influential observation in linear regression. *Technometrics*, 19(1), 15–18. https://doi.org/10.1080/00401706.1977.10489493

Cox, D. R. (1958). The regression analysis of binary sequences. *Journal of the Royal Statistical Society, Series B*, 20(2), 215–232. https://doi.org/10.1111/j.2517-6161.1958.tb00292.x

Davis, J. and Goadrich, M. (2006). The relationship between precision-recall and ROC curves. *Proceedings of the 23rd International Conference on Machine Learning*, 233–240. https://doi.org/10.1145/1143844.1143874

Hoerl, A. E. and Kennard, R. W. (1970). Ridge regression: biased estimation for nonorthogonal problems. *Technometrics*, 12(1), 55–67. https://doi.org/10.1080/00401706.1970.10488634

MacKinnon, J. G. and White, H. (1985). Some heteroskedasticity-consistent covariance matrix estimators with improved finite sample properties. *Journal of Econometrics*, 29(3), 305–325. https://doi.org/10.1016/0304-4076(85)90158-7

Marquardt, D. W. (1970). Generalized inverses, ridge regression, biased linear estimation, and nonlinear estimation. *Technometrics*, 12(3), 591–612. https://doi.org/10.1080/00401706.1970.10488699

Naeini, M. P., Cooper, G. F., and Hauskrecht, M. (2015). Obtaining well calibrated probabilities using Bayesian binning. *Proceedings of the 29th AAAI Conference on Artificial Intelligence*, 2901–2907. https://doi.org/10.1609/aaai.v29i1.9602

Saito, T. and Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE*, 10(3), e0118432. https://doi.org/10.1371/journal.pone.0118432

White, H. (1980). A heteroskedasticity-consistent covariance matrix estimator and a direct test for heteroskedasticity. *Econometrica*, 48(4), 817–838. https://doi.org/10.2307/1912934